In [1]:
import nest_asyncio
import re
from crawl4ai import AsyncWebCrawler, BrowserConfig, CrawlerRunConfig, CacheMode
from langchain_core.documents import Document

# This is the magic line for Jupyter notebooks
nest_asyncio.apply()

# urls = [
#     "https://www.bio-monitoring.ca",
#     "https://www.bio-monitoring.ca/scientific-technical-innovation",
#     "https://www.bio-monitoring.ca/about-us-1",
#     "https://www.bio-monitoring.ca/mission-vision",
#     "https://www.bio-monitoring.ca/contact",
#     "https://www.bio-monitoring.ca/product/21308262/folic-acid-supplement"
# ]
urls = [
    "https://propastry.ca/about/",
    "https://propastry.ca",
    'https://propastry.ca/cart/',
    'https://propastry.ca/contact/',
    'https://propastry.ca/faq/',
    'https://propastry.ca/resources/',
    'https://propastry.ca/services/',
    'https://propastry.ca/shop/'
]

async def load_web_docs_parallel(urls):
    browser_config = BrowserConfig(headless=True, verbose=False)
    run_config = CrawlerRunConfig(
        cache_mode=CacheMode.BYPASS,
        semaphore_count=5,
        excluded_tags=['nav'], # <nav>
        excluded_selector='.nav, .menu, [class^="menu"], [class^="nav"]', #<div class="menu">, starts with nav or starts with menu
        remove_overlay_elements=True
    )

    documents = []

    async with AsyncWebCrawler(config=browser_config) as crawler:
        # Use arun_many for parallel execution
        results = await crawler.arun_many(urls=urls, config=run_config)

        for result in results:
            if result.success:
                # Get the raw markdown
                content = result.markdown.raw_markdown 

                doc = Document(
                    page_content=content,
                    metadata={"source": result.url}
                )
                documents.append(doc)
            else:
                print(f"Error at {result.url}: {result.error_message}")

    return documents

# In Jupyter, you can use 'await' directly or call it like this:
docs = await load_web_docs_parallel(urls)

# View the first document's content
print(f"Loaded {len(docs)} documents.")

[FETCH]... ↓ https://propastry.ca/cart/                                                                           |
✓ | ⏱: 2.48s 

[SCRAPE].. ◆ https://propastry.ca/cart/                                                                           |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://propastry.ca/cart/                                                                           |
✓ | ⏱: 2.50s 

[FETCH]... ↓ https://propastry.ca/about/                                                                          |
✓ | ⏱: 2.64s 

[SCRAPE].. ◆ https://propastry.ca/about/                                                                          |
✓ | ⏱: 0.01s 

...

[COMPLETE] ● https://propastry.ca/contact/                                                                        |
✓ | ⏱: 2.73s 

[FETCH]... ↓ https://propastry.ca                                                                                 |
✓ | ⏱: 4.55s 

[SCRAPE].. ◆ https://propastry.ca                                                                                 |
✓ | ⏱: 0.01s 

[COMPLETE] ● https://propastry.ca                                                                                 |
✓ | ⏱: 4.56s 

Loaded 8 documents.


In [20]:
print(docs[7].page_content)#['page_content']


[![Propastry for health conscious mothers](https://propastry.ca/wp-content/uploads/2025/12/ext-custom-logo-1764806060388.png)](https://propastry.ca/)
# [Propastry for health conscious mothers](https://propastry.ca)
[ ](https://propastry.ca/my-account/)
0
  * [](https://www.instagram.com/)
  * [](https://www.facebook.com/)
  * [](https://x.com/)


# Nourishing Families with Every Bake
Empowering mothers to create wholesome, tasty treats
[Discover More](https://propastry.ca/services)
![](https://propastry.ca/wp-content/uploads/2025/12/e23e43830381e8d5af4e4e35ca8bedccd931ce58.jpg)
## Blog
Explore our collection of nutritious baking recipes, expert tips, and family-friendly ideas designed to support your healthy lifestyle.
  * [![Balancing Flavor and Nutrition in Every Bite](https://propastry.ca/wp-content/uploads/2025/12/featured-image-7.jpg)](https://propastry.ca/balancing-flavor-and-nutrition-in-every-bite/)
[Category 4](https://propastry.ca/category/category-4/)
## [Balancing Flavor an

In [13]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Chunk the text (VERY important for RAG)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100,
)

chunks = text_splitter.split_documents(docs)

print(f"Loaded {len(chunks)} chunks")

Loaded 30 chunks


In [21]:
import os
from langchain_community.vectorstores import UpstashVectorStore

with open('../upstash.txt', 'r') as f:
    lines = f.readlines()
# Set your credentials
os.environ["UPSTASH_VECTOR_REST_URL"] = lines[0].strip()
os.environ["UPSTASH_VECTOR_REST_TOKEN"] = lines[1].strip()
# name_space = 'bio-monitoring.ca'
name_space = 'propastry.ca'
# Path A: Using Upstash Hosted Embeddings (The "Better" way)
# We pass embedding=True so LangChain knows Upstash handles the vectorization
vectorstore = UpstashVectorStore(
    embedding=True, # Whether the embedding should be calculated in cloud
    namespace = name_space # Your first namespace
)

# Add your existing 'chunks' from your previous code
vectorstore.add_documents(chunks)

print(f"Docs added to namespace: {name_space}")

Docs added to namespace: propastry.ca


In [104]:
#Sanity check
query = "What is their innovation?"
results = vectorstore.similarity_search(query, k=2)

for i, doc in enumerate(results, 1):
    clean_string = " ".join(doc.page_content.split())
    print(f"\nResult {i}:\n{clean_string}...")



Result 1:
### -Producing natural, research-driven alternatives to synthetic supplements. ### -Advancing sustainable food biotechnology within Alberta’s bioeconomy. # VISION ## To become Alberta’s leading innovator in bio-based nutrition, creating a future to support human health, and reduce NTD births. **We produce highly bioactive, natural folic acid through precision fermentation, giving food, supplement, and pharma brands a cleaner, more effective folate ingredient that’s easier to absorb, more sustainable to manufacture, and fully aligned with modern regulatory and consumer expectations.** ### Chat Assistant Clear Chat...

Result 2:
## Our technology targets **clinically complex, high‑value segments** where standard synthetic folic acid may be suboptimal ## Our flagship initiative, the BioFolate Project, focuses on producing natural folic acid (vitamin B9) through fermentation using optimized _Lactobacillus_ strains. This approach reduces dependency on synthetic vitamins and suppo